# Amazon Earnings Transcript Scraper (Q1–Q4 FY2024)

This notebook scrapes earnings call transcripts for Amazon (Q1–Q4 FY2024) from Fool.com, cleans the raw text to remove disclaimers and noise, and saves each cleaned transcript to a separate file.
## Amazon's fiscal year runs from January 1st to December 31st.
**Steps Covered:**
1. Scrape earnings call transcripts using Selenium
2. Clean the scraped text
3. Save a single cleaned `.txt` file for each quarter

In [1]:
# Import libraries
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import matplotlib.pyplot as plt
import time  
import os
import re
import sys
from openai import OpenAI
import json
import io


# Step 1: Scrape Transcript from Fool.com

This function uses Selenium to open the transcript webpage, accept cookies, and extract all paragraph text from the main article body.


In [2]:
#Now, we run the scraping function
def scrape_clean_transcript(url):
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    driver.get(url)
    time.sleep(5)

    # Accept cookie banner if present
    try:
        driver.find_element(By.XPATH,'/html/body/div[13]/div[2]/div/div/div[2]/div/div/button[2]').click()
        time.sleep(1)
    except:
        pass

    # Extract the article content
    try:
        article_body = driver.find_element(By.XPATH, "/html/body/div[9]/div[3]/div[2]/section[2]/div/div[2]/div[1]/div[1]")
        paragraphs = article_body.find_elements(By.XPATH, "//p")
        raw_text = "\n".join([p.text for p in paragraphs if p.text.strip()])
    except Exception as e:
        print(f"Error extracting transcript from {url}: {e}")
        raw_text = ""

    driver.quit()
    return raw_text


# Step 2: Clean the Transcript Text

This function removes empty lines and disclaimer/legal language. It also collapses multiple line breaks into a single one.


In [3]:
def clean_transcript(text):
    lines = text.strip().split("\n")

    # 1. Remove leading "header" lines if they look like heading metadata
    cleaned_lines = []
    for line in lines:
        if (
            any(kw in line.lower() for kw in ["image source", "amazon", "earnings call", "et"]) and
            len(cleaned_lines) < 5
        ):
            continue
        cleaned_lines.append(line.strip())

    text = "\n".join(cleaned_lines).strip()

    # 2. Truncate known footer section
    footer_start = text.find("More AMZN analysis")
    if footer_start != -1:
        text = text[:footer_start].strip()

    # 3. Clean up legal terms and short lines
    final_lines = []
    for line in text.split("\n"):
        if len(line.strip()) < 5:
            continue
        if any(term in line.lower() for term in ["forward-looking", "safe harbor", "disclaimer"]):
            continue
        final_lines.append(line.strip())

    cleaned_text = "\n".join(final_lines)
    return re.sub(r'\n+', '\n', cleaned_text)



# Step 3: Save Cleaned Transcript to File

This function saves the cleaned transcript to a `.txt` file inside the `clean_transcripts/` directory.


In [4]:
def save_transcript(text, filename):
    os.makedirs("clean_transcripts", exist_ok=True)
    path = os.path.join("clean_transcripts", filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"Saved: {path}")

# Step 4: Process All Four Quarters (Q1–Q4 FY2025)

We define a dictionary of transcript URLs and filenames. The script loops through them, scrapes the data, cleans it, and saves the result.


In [5]:
transcripts = {
    "AMZN_q1_2024.txt": "https://www.fool.com/earnings/call-transcripts/2024/04/30/amazoncom-amzn-q1-2024-earnings-call-transcript/",
    "AMZN_q2_2024.txt": "https://www.fool.com/earnings/call-transcripts/2024/08/01/amazoncom-amzn-q2-2024-earnings-call-transcript/",
    "AMZN_q3_2024.txt": "https://www.fool.com/earnings/call-transcripts/2024/10/31/amazoncom-amzn-q3-2024-earnings-call-transcript/",
    "AMZN_q4_2024.txt": "https://www.fool.com/earnings/call-transcripts/2025/02/06/amazoncom-amzn-q4-2024-earnings-call-transcript/",
    "AMZN_q1_2023.txt": "https://www.fool.com/earnings/call-transcripts/2023/04/28/amazoncom-amzn-q1-2023-earnings-call-transcript/",
    "AMZN_q2_2023.txt": "https://www.fool.com/earnings/call-transcripts/2023/08/03/amazoncom-amzn-q2-2023-earnings-call-transcript/",
    "AMZN_q3_2023.txt": "https://www.fool.com/earnings/call-transcripts/2023/10/26/amazoncom-amzn-q3-2023-earnings-call-transcript/",
    "AMZN_q4_2023.txt": "https://www.fool.com/earnings/call-transcripts/2024/02/01/amazoncom-amzn-q4-2023-earnings-call-transcript/"
}

for filename, url in transcripts.items():
    print(f"\nProcessing {filename} ...")
    raw = scrape_clean_transcript(url)
    clean = clean_transcript(raw)
    save_transcript(clean, filename)



Processing AMZN_q1_2024.txt ...
Saved: clean_transcripts\AMZN_q1_2024.txt

Processing AMZN_q2_2024.txt ...
Saved: clean_transcripts\AMZN_q2_2024.txt

Processing AMZN_q3_2024.txt ...
Saved: clean_transcripts\AMZN_q3_2024.txt

Processing AMZN_q4_2024.txt ...
Saved: clean_transcripts\AMZN_q4_2024.txt

Processing AMZN_q1_2023.txt ...
Saved: clean_transcripts\AMZN_q1_2023.txt

Processing AMZN_q2_2023.txt ...
Saved: clean_transcripts\AMZN_q2_2023.txt

Processing AMZN_q3_2023.txt ...
Saved: clean_transcripts\AMZN_q3_2023.txt

Processing AMZN_q4_2023.txt ...
Saved: clean_transcripts\AMZN_q4_2023.txt


# Step 5: Preprocess the Cleaned Transcripts

In [6]:
# Define the input transcript files and associated quarters
transcript_files = {
    'Q1_2024': 'clean_transcripts/AMZN_q1_2024.txt',
    'Q2_2024': 'clean_transcripts/AMZN_q2_2024.txt',
    'Q3_2024': 'clean_transcripts/AMZN_q3_2024.txt',
    'Q4_2024': 'clean_transcripts/AMZN_q4_2024.txt',
    'Q1_2023': 'clean_transcripts/AMZN_q1_2023.txt',
    'Q2_2023' : 'clean_transcripts/AMZN_q2_2023.txt',
    'Q3_2023' : 'clean_transcripts/AMZN_q3_2023.txt',
    'Q4_2023' : 'clean_transcripts/AMZN_q4_2023.txt'
}

records = []

# Improved regex pattern for speaker lines
speaker_pattern = re.compile(
    r'^(?P<speaker>([A-Z][a-zA-Z.,\'-]+\s){1,3}[A-Z][a-zA-Z.,\'-]+)\s--\s(?P<title>[A-Za-z][^:\n]+)$'
)

for quarter, path in transcript_files.items():
    with open(path, 'r', encoding='utf-8') as f:
        current_speaker = None
        current_title = None
        buffer = []

        for raw_line in f:
            line = raw_line.strip()

            # Remove notes like [Operator Instructions]
            line = re.sub(r'\[.*?\]', '', line).strip()
            if not line:
                continue

            match = speaker_pattern.match(line)
            if match:
                # Save previous speaker block
                if current_speaker and buffer:
                    content = ' '.join(buffer).strip()
                    if len(content) > 30:
                        records.append({
                            'quarter': quarter,
                            'speaker': current_speaker,
                            'title': current_title,
                            'content': content
                        })

                # Start new block
                current_speaker = match.group('speaker').strip()
                current_title = match.group('title').strip()
                buffer = []

            else:
                # If line is misparsed (e.g. a regular sentence starting with "And I..."), just append to buffer
                buffer.append(line)

        # Flush the last block
        if current_speaker and buffer:
            content = ' '.join(buffer).strip()
            if len(content) > 30:
                records.append({
                    'quarter': quarter,
                    'speaker': current_speaker,
                    'title': current_title,
                    'content': content
                })

# Save to DataFrame
df = pd.DataFrame(records)

# Optional: forward-fill missing speakers and titles (if you later add edge case handling)
df['speaker'].ffill(inplace=True)
df['title'].ffill(inplace=True)

# Save to CSV
output_path = 'clean_transcripts/AMAZON_all_quarters_speaker_blocks.csv'
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} speaker blocks to {output_path}")


df['speaker'].ffill(inplace=True)
if 'title' in df.columns:
    df['title'].ffill(inplace=True)

# Save cleaned speaker blocks
output_path = 'clean_transcripts/AMAZON_all_quarters_speaker_blocks.csv'
df.to_csv(output_path, index=False)

print(f"Saved {len(df)} speaker blocks to {output_path}")


Saved 143 speaker blocks to clean_transcripts/AMAZON_all_quarters_speaker_blocks.csv
Saved 143 speaker blocks to clean_transcripts/AMAZON_all_quarters_speaker_blocks.csv


C:\Users\Tithi\AppData\Local\Temp\ipykernel_14808\1578803083.py:71: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['speaker'].ffill(inplace=True)
C:\Users\Tithi\AppData\Local\Temp\ipykernel_14808\1578803083.py:72: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when do

In [7]:
# Load the uploaded CSV file containing all quarters' speaker blocks
df_speaker_blocks = pd.read_csv("clean_transcripts/AMAZON_all_quarters_speaker_blocks.csv")

# Display the first few rows to confirm structure
df_speaker_blocks.head(8)

,quarter,speaker,title,content
0,Q1_2024,Dave Fildes,"Director, Investor Relations",It's not possible to accurately predict demand...
1,Q1_2024,Andy Jassy,Chief Executive Officer,"Thanks, Dave. Today, we're reporting $143.3 bi..."
2,Q1_2024,Brian Olsavsky,Chief Financial Officer,"Thanks, Andy. Starting with our top line finan..."
3,Q1_2024,Doug Anmuth,JPMorgan Chase and Company -- Analyst,Thanks so much for taking the question. Probab...
4,Q1_2024,Brian Olsavsky,Chief Financial Officer,"Hi. Doug, yeah, we have historically always me..."
5,Q1_2024,Andy Jassy,Chief Executive Officer,"Yeah, I just would add briefly, just to summar..."
6,Q1_2024,Ross Sandler,Barclays -- Analyst,"Hey, guys. Somewhat related question on capex ..."
7,Q1_2024,Andy Jassy,Chief Executive Officer,"Well, Ross, I would tell you that we have seen..."


# Step 6: Speaker-level Sentiment Analysis

In [8]:
from transformers import BertTokenizer, BertForSequenceClassification, pipeline
import pandas as pd

# Load your speaker blocks file (all quarters)
df = pd.read_csv("clean_transcripts/AMAZON_all_quarters_speaker_blocks.csv")

# FinBERT model setup
model_name = "ProsusAI/finbert"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)
finbert = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

import textwrap

def get_sentiment_chunks(text, max_chunk_words=200):
    if not isinstance(text, str) or not text.strip():
        return [("EMPTY", 0.0)]

    # Split text into words, then wrap into word chunks
    words = text.split()
    chunks = [' '.join(words[i:i + max_chunk_words]) for i in range(0, len(words), max_chunk_words)]

    results = []
    for chunk in chunks:
        try:
            result = finbert(chunk)[0]  # Let pipeline handle tokenization
            results.append((result["label"], result["score"]))
        except:
            results.append(("ERROR", 0.0))

    return results



# Apply FinBERT to each content (in chunks)
df["sentiment_chunks"] = df["content"].apply(get_sentiment_chunks)

# Aggregation function (majority vote + max score for top label)
def aggregate_sentiment(chunk_list):
    if not chunk_list or chunk_list == [("EMPTY", 0.0)]:
        return "EMPTY", 0.0

    from collections import Counter
    label_counts = Counter([label for label, _ in chunk_list])
    top_label = label_counts.most_common(1)[0][0]
    max_conf = max(score for label, score in chunk_list if label == top_label)
    return top_label, max_conf

# Apply aggregation
df[["sentiment", "confidence"]] = df["sentiment_chunks"].apply(lambda x: pd.Series(aggregate_sentiment(x)))


def map_five_categories(row):
    label = row['sentiment'].lower()
    score = row['confidence']

    if label == 'positive':
        return 'Strong Positive' if score > 0.85 else 'Slightly Positive'
    elif label == 'negative':
        return 'Strong Negative' if score > 0.85 else 'Slightly Negative'
    elif label == 'neutral':
        return 'Neutral'
    else:
        return 'Uncertain'

df['sentiment_category'] = df.apply(map_five_categories, axis=1)

# Optional: re-order columns
df = df[['quarter', 'speaker', 'title', 'content', 'sentiment', 'confidence', 'sentiment_category']]

# Save output
df.to_csv("clean_transcripts/AMAZON_finbert_output.csv", index=False)


d:\Bayes_Software_Setups\Anaconda_Setup\envs\ARPTech\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu


# Step 7: using LLMs to get sentiment score for each block

## API call

In [9]:
api_key = 'sk-58c0df73519c42debe27d41e164d455a'
base_url = "https://api.deepseek.com"
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")
seed = 42

## Getting the sentiment score for each blocks of text using an LLM.

In [10]:
df_llm = pd.read_csv("clean_transcripts/AMAZON_finbert_output.csv")
df_llm = df_llm.reset_index(drop=False).rename(columns={"index":"id"})
payload = df_llm[["id","content"]].rename(columns={"content":"text"}).to_dict(orient="records")
# Make a fixed-size chunk generator
def chunk(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

# Define the prompt
system_prompt = """
You are a sentiment-analysis engine.
I will give you a JSON array of objects like:
  [ { "id": 0, "text": "…"}, { "id": 1, "text": "…"}, … ]

For each text, return a probability distribution over the five sentiment classes:
Strong Negative, Slightly Negative, Neutral, Slightly Positive, Strong Positive.
Label your columns:

id,
LLM_sentiment, 
LLM_pct_strong_positive,
LLM_pct_slightly_positive,
LLM_pct_neutral,
LLM_pct_slightly_negative,
LLM_pct_strong_negative

—with each LLM_pct_* a float from 0.0–1.0 summing to 1.0, one row per paragraph,
no markdown fences or commentary, pure CSV.
-for sentiment, choose one of the five classes. 
"""

# Function to run a batch of rows through the LLM
def run_batch(batch_rows):
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": json.dumps(batch_rows)}
        ],
        temperature=0.0,
        top_p=1.0
    )
    csv_str = resp.choices[0].message.content.strip()
    return pd.read_csv(io.StringIO(csv_str))

all_parts = []
for b in chunk(payload, 50):          # batches of 50 rows
    df_part = run_batch(b)            # call the LLM
    all_parts.append(df_part)

sent_df = pd.concat(all_parts, ignore_index=True)


## Merging with the original df

In [11]:
# Merge with the sentiment results
df_with_sentiment = df_llm.merge(
    sent_df,            # contains id, sentiment, pct_*
    on='id',            # join key
    how='left'          # keep every original row
)

df_with_sentiment.drop(columns=['id'], inplace=True)  # drop the id column

print(f"Successfully merged sentiment data for {len(df_with_sentiment)} rows")
df_with_sentiment.head()

Successfully merged sentiment data for 143 rows


,quarter,speaker,title,content,sentiment,confidence,sentiment_category,LLM_sentiment,LLM_pct_strong_positive,LLM_pct_slightly_positive,LLM_pct_neutral,LLM_pct_slightly_negative,LLM_pct_strong_negative
0,Q1_2024,Dave Fildes,"Director, Investor Relations",It's not possible to accurately predict demand...,neutral,0.782457,Neutral,Neutral,0.10,0.20,0.50,0.15,0.05
1,Q1_2024,Andy Jassy,Chief Executive Officer,"Thanks, Dave. Today, we're reporting $143.3 bi...",neutral,0.909178,Neutral,Slightly Positive,0.20,0.40,0.30,0.08,0.02
2,Q1_2024,Brian Olsavsky,Chief Financial Officer,"Thanks, Andy. Starting with our top line finan...",positive,0.953807,Strong Positive,Neutral,0.15,0.25,0.45,0.10,0.05
3,Q1_2024,Doug Anmuth,JPMorgan Chase and Company -- Analyst,Thanks so much for taking the question. Probab...,neutral,0.871681,Neutral,Neutral,0.10,0.20,0.50,0.15,0.05
4,Q1_2024,Brian Olsavsky,Chief Financial Officer,"Hi. Doug, yeah, we have historically always me...",positive,0.901178,Strong Positive,Neutral,0.15,0.25,0.45,0.10,0.05


In [12]:
#save df_with_sentiment to csv
output_path = 'clean_transcripts/AMAZON_finbert_deepseek_output.csv'
df_with_sentiment.to_csv(output_path, index=False)
print(f"Saved sentiment output to {output_path}")

Saved sentiment output to clean_transcripts/AMAZON_finbert_deepseek_output.csv


In [13]:
AMAZON_final = pd.read_csv("AMAZON_full.csv")

# Select only the required columns
AMAZON_final = AMAZON_final[[
    "quarter",
    "content",
    "sentiment_category",
    "title",
    "LLM_sentiment",
    "Manual Annotation"
]]

# Rename columns
AMAZON_final = AMAZON_final.rename(columns={
    "sentiment_category": "FinBERT_sentiment",
    "Manual Annotation": "manual_annotation"
})

# Rearrange columns so role_category comes before FinBERT_sentiment
AMAZON_final = AMAZON_final[[
    "quarter",
    "content",
    "title",
    "FinBERT_sentiment",
    "LLM_sentiment",
    "manual_annotation"
]]

# Check the updated dataframe
print(AMAZON_final)

     quarter                                            content  \
0    Q1_2024  They just revealed what they believe are the 1...   
1    Q1_2024  Thanks, Dave. Today, we're reporting $143.3 bi...   
2    Q1_2024  Thanks, Andy. Starting with our top line finan...   
3    Q1_2024  Thanks so much for taking the question. Probab...   
4    Q1_2024  Hi. Doug, yeah, we have historically always me...   
..       ...                                                ...   
138  Q4_2023  Thanks, everyone. I have one on grocery and on...   
139  Q4_2023  Yeah. On grocery, we're pleased with the progr...   
140  Q4_2023  Thanks very much. I just wanted to follow up o...   
141  Q4_2023  Yeah. So, Colin, I would say a few things on -...   
142  Q4_2023  Thanks for joining us today on the call and fo...   

                                      title FinBERT_sentiment  \
0              Director, Investor Relations           Neutral   
1                   Chief Executive Officer           Neutral   
